
# MMRL vs BayesMMRL 特征分析（修正版）

这个 notebook 用来分析：

1. BayesMMRL 为什么可能有效  
2. BayesMMRL 内部特征如何变化  
3. 相对 MMRL，特征发生了什么变化  

它通过统一的 adapter 层兼容 MMRL 与 BayesMMRL 常见的不同推理接口，重点支持：
- `forward_eval(...)`
- `representation_learner.project_mean_tokens()`
- `representation_learner.project_sample_tokens()`
- `_encode_with_tokens(...)`

你只需要在“用户配置”和“手动适配”单元中填好项目路径、checkpoint 和 dataloader / model 构造方式，后面的分析部分可以直接复用。


In [2]:

from pathlib import Path

PROJECT_ROOT = Path("/path/to/your/project")   # TODO
MMRL_CKPT = Path("/path/to/mmrl_checkpoint.pth")  # TODO
BAYES_CKPT = Path("/path/to/bayes_mmrl_checkpoint.pth")  # TODO

MMRL_CFG = None
BAYES_CFG = None

DEVICE = "cuda"
BATCH_LIMIT = None
N_MC_BAYES = 10
BAYES_EVAL_MODE = "mc_only"   # "mc_only" | "mean_only" | "mean_plus_mc"
CLASSNAMES = None
USE_MANUAL_BUILDERS = True    # 没有统一自动入口时，建议直接 True


In [3]:

import os
import sys
import math
import copy
import json
import types
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python path injected:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python path injected: /path/to/your/project
Torch: 2.4.0+cu121
CUDA available: True


## 手动适配（只改这一格）

In [4]:

# 自动加载失败或不想依赖框架时，在这里实现三个函数。
# 要求：
#   build_mmrl_model() -> 已加载 checkpoint 且 .eval() 的 MMRL 模型
#   build_bayes_model() -> 已加载 checkpoint 且 .eval() 的 BayesMMRL 模型
#   build_eval_loader() -> 迭代返回 (image, label) 或 {"img"/"image":..., "label":...}

def build_mmrl_model():
    raise NotImplementedError("请在这里返回已加载好的 MMRL 模型")

def build_bayes_model():
    raise NotImplementedError("请在这里返回已加载好的 BayesMMRL 模型")

def build_eval_loader():
    raise NotImplementedError("请在这里返回评估 dataloader")


## 工具函数

In [5]:

def move_to_device(x, device=DEVICE):
    if torch.is_tensor(x):
        return x.to(device)
    if isinstance(x, (list, tuple)):
        return type(x)(move_to_device(v, device) for v in x)
    if isinstance(x, dict):
        return {k: move_to_device(v, device) for k, v in x.items()}
    return x

def batch_to_xy(batch):
    if isinstance(batch, (list, tuple)):
        if len(batch) < 2:
            raise ValueError("tuple/list batch 至少应包含 (image, label)")
        return batch[0], batch[1]
    if isinstance(batch, dict):
        img = batch.get("img", batch.get("image", batch.get("images", None)))
        label = batch.get("label", batch.get("labels", batch.get("target", None)))
        if img is None or label is None:
            raise KeyError("dict batch 中未找到 image/label")
        return img, label
    raise TypeError(f"Unsupported batch type: {type(batch)}")

def top1_conf_margin(logits):
    prob = torch.softmax(logits, dim=-1)
    conf, pred = prob.max(dim=-1)
    top2 = torch.topk(prob, k=min(2, prob.shape[-1]), dim=-1).values
    margin = top2[:, 0] if top2.shape[-1] == 1 else top2[:, 0] - top2[:, 1]
    return pred, conf, margin, prob

def cosine_per_sample(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return (a * b).sum(dim=-1)

def sym_kl_from_logits(logits_p, logits_q, eps=1e-8):
    p = torch.softmax(logits_p, dim=-1).clamp_min(eps)
    q = torch.softmax(logits_q, dim=-1).clamp_min(eps)
    kl_pq = (p * (p.log() - q.log())).sum(dim=-1)
    kl_qp = (q * (q.log() - p.log())).sum(dim=-1)
    return 0.5 * (kl_pq + kl_qp)

def js_div_from_logits(logits_p, logits_q, eps=1e-8):
    p = torch.softmax(logits_p, dim=-1).clamp_min(eps)
    q = torch.softmax(logits_q, dim=-1).clamp_min(eps)
    m = 0.5 * (p + q)
    js = 0.5 * (p * (p.log() - m.log())).sum(dim=-1) + 0.5 * (q * (q.log() - m.log())).sum(dim=-1)
    return js

def predictive_entropy_from_probs(probs, eps=1e-8):
    probs = probs.clamp_min(eps)
    return -(probs * probs.log()).sum(dim=-1)

def mutual_information_from_mc_probs(mc_probs, eps=1e-8):
    mean_probs = mc_probs.mean(dim=0)
    pred_ent = predictive_entropy_from_probs(mean_probs, eps=eps)
    exp_ent = predictive_entropy_from_probs(mc_probs, eps=eps).mean(dim=0)
    return pred_ent, exp_ent, pred_ent - exp_ent

def maybe_set_bayes_eval_mode(model, mode):
    if hasattr(model, "eval_mode"):
        old = model.eval_mode
        model.eval_mode = mode
        return old
    return None

def restore_bayes_eval_mode(model, old):
    if old is not None and hasattr(model, "eval_mode"):
        model.eval_mode = old

def aggregate_sample_dicts(samples):
    keys = samples[0].keys()
    agg = {}
    for k in keys:
        vals = [s[k] for s in samples if k in s and torch.is_tensor(s[k])]
        if len(vals) > 0:
            agg[k] = torch.stack(vals, dim=0).mean(dim=0)
    return agg


## 模型适配器

In [6]:

class ModelAdapter:
    def __init__(self, model, name="model"):
        self.model = model
        self.name = name

    def _normalize_output(self, out):
        # 统一成 dict: logits_c, logits_r, logits_f, feat_c, text_feat
        if isinstance(out, dict):
            result = dict(out)
        elif isinstance(out, (list, tuple)):
            if len(out) >= 5:
                logits_c, logits_r, logits_f, feat_c, text_feat = out[:5]
                result = {
                    "logits_c": logits_c,
                    "logits_r": logits_r,
                    "logits_f": logits_f,
                    "feat_c": feat_c,
                    "text_feat": text_feat,
                }
            else:
                raise ValueError(f"{self.name}: 无法解析输出 tuple，长度={len(out)}")
        else:
            raise TypeError(f"{self.name}: Unsupported output type {type(out)}")

        aliases = {
            "logits_c": ["logits", "main_logits", "cls_logits"],
            "logits_r": ["rep_logits", "logits_rep"],
            "logits_f": ["fusion_logits", "logits_fusion"],
            "feat_c": ["image_features", "feat_main", "cls_features"],
            "text_feat": ["text_features", "txt_features"],
            "feat_r": ["image_features_rep", "feat_rep", "rep_features"],
        }
        for target, names in aliases.items():
            if target not in result:
                for k in names:
                    if k in result:
                        result[target] = result[k]
                        break
        return result

    @torch.no_grad()
    def mean_forward(self, image):
        model = self.model
        model.eval()
        if hasattr(model, "forward_eval"):
            old = maybe_set_bayes_eval_mode(model, "mean_only")
            try:
                out = model.forward_eval(image, num_samples=1)
            finally:
                restore_bayes_eval_mode(model, old)
            return self._normalize_output(out)

        if hasattr(model, "forward") and callable(model.forward):
            out = model(image)
            return self._normalize_output(out)

        raise RuntimeError(f"{self.name}: 无可用推理接口")

    @torch.no_grad()
    def mc_forward(self, image, num_samples=10, include_mean=False):
        model = self.model
        model.eval()

        if hasattr(model, "forward_eval"):
            samples = []

            if include_mean:
                old = maybe_set_bayes_eval_mode(model, "mean_plus_mc")
                try:
                    agg = self._normalize_output(model.forward_eval(image, num_samples=num_samples))
                finally:
                    restore_bayes_eval_mode(model, old)

                old = maybe_set_bayes_eval_mode(model, "mean_only")
                try:
                    samples.append(self._normalize_output(model.forward_eval(image, num_samples=1)))
                finally:
                    restore_bayes_eval_mode(model, old)

                old = maybe_set_bayes_eval_mode(model, "mc_only")
                try:
                    for _ in range(max(0, num_samples - 1)):
                        samples.append(self._normalize_output(model.forward_eval(image, num_samples=1)))
                finally:
                    restore_bayes_eval_mode(model, old)
                return {"samples": samples, "agg": agg}

            old = maybe_set_bayes_eval_mode(model, "mc_only")
            try:
                for _ in range(num_samples):
                    samples.append(self._normalize_output(model.forward_eval(image, num_samples=1)))
            finally:
                restore_bayes_eval_mode(model, old)
            return {"samples": samples, "agg": aggregate_sample_dicts(samples)}

        base = self.mean_forward(image)
        samples = [base for _ in range(num_samples)]
        return {"samples": samples, "agg": aggregate_sample_dicts(samples)}


## 构建模型与 loader

In [7]:

mmrl_model = build_mmrl_model()
bayes_model = build_bayes_model()
eval_loader = build_eval_loader()

mmrl_model = mmrl_model.to(DEVICE).eval()
bayes_model = bayes_model.to(DEVICE).eval()

mmrl = ModelAdapter(mmrl_model, "MMRL")
bayes = ModelAdapter(bayes_model, "BayesMMRL")

print("MMRL model ready:", type(mmrl_model))
print("Bayes model ready:", type(bayes_model))
print("Eval loader ready:", type(eval_loader))


NotImplementedError: 请在这里返回已加载好的 MMRL 模型

## 核心分析逻辑

In [ ]:

@torch.no_grad()
def analyze_one_batch(adapter, image, label, n_mc=10, bayes_include_mean=False):
    image = move_to_device(image, DEVICE)
    label = move_to_device(label, DEVICE)

    mean_out = adapter.mean_forward(image)

    zc = mean_out["logits_c"]
    zr = mean_out["logits_r"]
    zf = mean_out["logits_f"]
    fc = mean_out.get("feat_c", None)
    fr = mean_out.get("feat_r", None)

    pred_c, conf_c, margin_c, prob_c = top1_conf_margin(zc)
    pred_r, conf_r, margin_r, prob_r = top1_conf_margin(zr)
    pred_f, conf_f, margin_f, prob_f = top1_conf_margin(zf)

    correct_c = pred_c.eq(label)
    correct_r = pred_r.eq(label)
    correct_f = pred_f.eq(label)

    feat_cos = cosine_per_sample(fc, fr).detach().cpu() if (torch.is_tensor(fc) and torch.is_tensor(fr)) else cosine_per_sample(zc, zr).detach().cpu()

    row = {
        "y": label.detach().cpu(),
        "pred_c": pred_c.detach().cpu(),
        "pred_r": pred_r.detach().cpu(),
        "pred_f": pred_f.detach().cpu(),
        "correct_c": correct_c.detach().cpu(),
        "correct_r": correct_r.detach().cpu(),
        "correct_f": correct_f.detach().cpu(),
        "conf_c": conf_c.detach().cpu(),
        "conf_r": conf_r.detach().cpu(),
        "conf_f": conf_f.detach().cpu(),
        "margin_c": margin_c.detach().cpu(),
        "margin_r": margin_r.detach().cpu(),
        "margin_f": margin_f.detach().cpu(),
        "feat_cos": feat_cos,
        "logit_cos": cosine_per_sample(zc, zr).detach().cpu(),
        "sym_kl": sym_kl_from_logits(zc, zr).detach().cpu(),
        "js_div": js_div_from_logits(zc, zr).detach().cpu(),
        "c_only": (correct_c & (~correct_r)).detach().cpu(),
        "r_only": (correct_r & (~correct_c)).detach().cpu(),
        "fusion_beats_c": (correct_f & (~correct_c)).detach().cpu(),
        "fusion_beats_r": (correct_f & (~correct_r)).detach().cpu(),
        "fusion_beats_both": (correct_f & (~correct_c) & (~correct_r)).detach().cpu(),
        "hurt_c": ((~correct_f) & correct_c).detach().cpu(),
        "hurt_r": ((~correct_f) & correct_r).detach().cpu(),
        "hurt_best_branch": ((~correct_f) & (correct_c | correct_r)).detach().cpu(),
    }

    mc_pack = adapter.mc_forward(image, num_samples=n_mc, include_mean=bayes_include_mean)
    mc_samples = mc_pack["samples"]

    zc_mc = torch.stack([s["logits_c"] for s in mc_samples], dim=0)
    zr_mc = torch.stack([s["logits_r"] for s in mc_samples], dim=0)
    zf_mc = torch.stack([s["logits_f"] for s in mc_samples], dim=0)

    pf_mc = torch.softmax(zf_mc, dim=-1)

    row["logit_std_c"] = zc_mc.std(dim=0).mean(dim=-1).detach().cpu()
    row["logit_std_r"] = zr_mc.std(dim=0).mean(dim=-1).detach().cpu()
    row["logit_std_f"] = zf_mc.std(dim=0).mean(dim=-1).detach().cpu()

    pred_ent_f, exp_ent_f, mi_f = mutual_information_from_mc_probs(pf_mc)
    row["pred_entropy_f"] = pred_ent_f.detach().cpu()
    row["expected_entropy_f"] = exp_ent_f.detach().cpu()
    row["mutual_info_f"] = mi_f.detach().cpu()

    return row

def rows_to_df(rows):
    merged = {}
    for k in rows[0].keys():
        vals = [r[k] for r in rows]
        if torch.is_tensor(vals[0]):
            merged[k] = torch.cat([v.reshape(-1) for v in vals], dim=0).numpy()
        else:
            merged[k] = np.concatenate(vals)
    df = pd.DataFrame(merged)

    bool_cols = [
        "correct_c","correct_r","correct_f",
        "c_only","r_only",
        "fusion_beats_c","fusion_beats_r","fusion_beats_both",
        "hurt_c","hurt_r","hurt_best_branch"
    ]
    for c in bool_cols:
        if c in df.columns:
            df[c] = df[c].astype(bool)
    df["complementarity"] = df["c_only"].astype(int) + df["r_only"].astype(int)
    return df

def run_analysis(adapter, loader, n_mc=10, batch_limit=None, bayes_include_mean=False):
    rows = []
    for bi, batch in enumerate(loader):
        if batch_limit is not None and bi >= batch_limit:
            break
        image, label = batch_to_xy(batch)
        rows.append(analyze_one_batch(adapter, image, label, n_mc=n_mc, bayes_include_mean=bayes_include_mean))
        if (bi + 1) % 10 == 0:
            print(f"[{adapter.name}] processed {bi+1} batches")
    return rows_to_df(rows)


In [ ]:

mmrl_df = run_analysis(mmrl, eval_loader, n_mc=max(2, N_MC_BAYES), batch_limit=BATCH_LIMIT, bayes_include_mean=False)
bayes_df = run_analysis(bayes, eval_loader, n_mc=N_MC_BAYES, batch_limit=BATCH_LIMIT, bayes_include_mean=(BAYES_EVAL_MODE == "mean_plus_mc"))

print("MMRL samples:", len(mmrl_df))
print("Bayes samples:", len(bayes_df))
mmrl_df.head(), bayes_df.head()


## 汇总与对比

In [ ]:

def summarize_df(df, name="model"):
    return pd.Series({
        "name": name,
        "n": len(df),
        "acc_c": df["correct_c"].mean(),
        "acc_r": df["correct_r"].mean(),
        "acc_f": df["correct_f"].mean(),
        "conf_c": df["conf_c"].mean(),
        "conf_r": df["conf_r"].mean(),
        "conf_f": df["conf_f"].mean(),
        "margin_c": df["margin_c"].mean(),
        "margin_r": df["margin_r"].mean(),
        "margin_f": df["margin_f"].mean(),
        "feat_cos": df["feat_cos"].mean(),
        "logit_cos": df["logit_cos"].mean(),
        "sym_kl": df["sym_kl"].mean(),
        "js_div": df["js_div"].mean(),
        "complementarity": df["complementarity"].mean(),
        "fusion_beats_c": df["fusion_beats_c"].mean(),
        "fusion_beats_r": df["fusion_beats_r"].mean(),
        "fusion_beats_both": df["fusion_beats_both"].mean(),
        "hurt_c": df["hurt_c"].mean(),
        "hurt_r": df["hurt_r"].mean(),
        "hurt_best_branch": df["hurt_best_branch"].mean(),
        "logit_std_c": df["logit_std_c"].mean(),
        "logit_std_r": df["logit_std_r"].mean(),
        "logit_std_f": df["logit_std_f"].mean(),
        "pred_entropy_f": df["pred_entropy_f"].mean(),
        "expected_entropy_f": df["expected_entropy_f"].mean(),
        "mutual_info_f": df["mutual_info_f"].mean(),
    })

summary_table = pd.concat([
    summarize_df(mmrl_df, "MMRL"),
    summarize_df(bayes_df, "BayesMMRL"),
], axis=1).T

summary_table


In [ ]:

delta = summary_table.set_index("name").loc["BayesMMRL"] - summary_table.set_index("name").loc["MMRL"]
delta = delta.to_frame("Bayes - MMRL")
delta


## 可视化

In [ ]:

def plot_metric_compare(df_a, df_b, col, name_a="MMRL", name_b="BayesMMRL", bins=40):
    plt.figure(figsize=(6,4))
    plt.hist(df_a[col], bins=bins, alpha=0.5, density=True, label=name_a)
    plt.hist(df_b[col], bins=bins, alpha=0.5, density=True, label=name_b)
    plt.title(col)
    plt.legend()
    plt.tight_layout()
    plt.show()

for metric in [
    "feat_cos", "logit_cos", "sym_kl", "js_div",
    "conf_f", "margin_f",
    "logit_std_f", "pred_entropy_f", "mutual_info_f"
]:
    plot_metric_compare(mmrl_df, bayes_df, metric)


## BayesMMRL 为什么有效

In [ ]:

interpret_rows = []

def add_interpretation(title, current, why_good, why_bad):
    interpret_rows.append({
        "现象": title,
        "当前值": current,
        "若 Bayes 更优时常见解释": why_good,
        "若 Bayes 更差时常见解释": why_bad,
    })

add_interpretation(
    "fusion acc",
    f"{summary_table.loc[1, 'acc_f']:.4f} vs {summary_table.loc[0, 'acc_f']:.4f}",
    "Bayes 对 rep 分支做采样平均，使 fusion 更像稳健纠错，而不是过拟合偏置放大器。",
    "stochastic rep 没提供有效补充，噪声大于信息。"
)

add_interpretation(
    "hurt_best_branch",
    f"{summary_table.loc[1, 'hurt_best_branch']:.4f} vs {summary_table.loc[0, 'hurt_best_branch']:.4f}",
    "Bayes fusion 更少把原本正确样本拉错，是稳定性的关键证据。",
    "fusion 仍在引入额外噪声。"
)

add_interpretation(
    "complementarity",
    f"{summary_table.loc[1, 'complementarity']:.4f} vs {summary_table.loc[0, 'complementarity']:.4f}",
    "若保持或略升，说明 Bayes 没让两分支塌缩。",
    "若显著下降，可能说明 rep 分支被过度收缩。"
)

add_interpretation(
    "mutual_info_f",
    f"{summary_table.loc[1, 'mutual_info_f']:.4f} vs {summary_table.loc[0, 'mutual_info_f']:.4f}",
    "适度 MI 表示能识别 epistemic uncertainty。",
    "过低说明退化成弱噪声注入；过高说明采样过强。"
)

pd.DataFrame(interpret_rows)


## 内部特征如何变化

In [ ]:

internal_change = pd.DataFrame({
    "指标": [
        "feat_cos", "logit_cos", "sym_kl", "js_div",
        "logit_std_r", "logit_std_f", "mutual_info_f",
        "fusion_beats_both", "hurt_best_branch",
    ],
    "MMRL": [
        summary_table.loc[0, "feat_cos"],
        summary_table.loc[0, "logit_cos"],
        summary_table.loc[0, "sym_kl"],
        summary_table.loc[0, "js_div"],
        summary_table.loc[0, "logit_std_r"],
        summary_table.loc[0, "logit_std_f"],
        summary_table.loc[0, "mutual_info_f"],
        summary_table.loc[0, "fusion_beats_both"],
        summary_table.loc[0, "hurt_best_branch"],
    ],
    "BayesMMRL": [
        summary_table.loc[1, "feat_cos"],
        summary_table.loc[1, "logit_cos"],
        summary_table.loc[1, "sym_kl"],
        summary_table.loc[1, "js_div"],
        summary_table.loc[1, "logit_std_r"],
        summary_table.loc[1, "logit_std_f"],
        summary_table.loc[1, "mutual_info_f"],
        summary_table.loc[1, "fusion_beats_both"],
        summary_table.loc[1, "hurt_best_branch"],
    ],
})
internal_change["Bayes-MMRL"] = internal_change["BayesMMRL"] - internal_change["MMRL"]
internal_change


## 相对 MMRL，特征变化情况

In [ ]:

compare_df = pd.DataFrame({
    "mmrl_correct_f": mmrl_df["correct_f"].values,
    "bayes_correct_f": bayes_df["correct_f"].values,
    "mmrl_conf_f": mmrl_df["conf_f"].values,
    "bayes_conf_f": bayes_df["conf_f"].values,
    "mmrl_margin_f": mmrl_df["margin_f"].values,
    "bayes_margin_f": bayes_df["margin_f"].values,
    "mmrl_feat_cos": mmrl_df["feat_cos"].values,
    "bayes_feat_cos": bayes_df["feat_cos"].values,
    "mmrl_logit_cos": mmrl_df["logit_cos"].values,
    "bayes_logit_cos": bayes_df["logit_cos"].values,
    "mmrl_sym_kl": mmrl_df["sym_kl"].values,
    "bayes_sym_kl": bayes_df["sym_kl"].values,
    "mmrl_js_div": mmrl_df["js_div"].values,
    "bayes_js_div": bayes_df["js_div"].values,
    "mmrl_logit_std_f": mmrl_df["logit_std_f"].values,
    "bayes_logit_std_f": bayes_df["logit_std_f"].values,
    "mmrl_mutual_info_f": mmrl_df["mutual_info_f"].values,
    "bayes_mutual_info_f": bayes_df["mutual_info_f"].values,
})

compare_df["bayes_better"] = (~compare_df["mmrl_correct_f"]) & (compare_df["bayes_correct_f"])
compare_df["bayes_worse"] = (compare_df["mmrl_correct_f"]) & (~compare_df["bayes_correct_f"])
compare_df["same_correct"] = (compare_df["mmrl_correct_f"]) & (compare_df["bayes_correct_f"])
compare_df["same_wrong"] = (~compare_df["mmrl_correct_f"]) & (~compare_df["bayes_correct_f"])

for name, a, b in [
    ("delta_conf_f", "bayes_conf_f", "mmrl_conf_f"),
    ("delta_margin_f", "bayes_margin_f", "mmrl_margin_f"),
    ("delta_feat_cos", "bayes_feat_cos", "mmrl_feat_cos"),
    ("delta_logit_cos", "bayes_logit_cos", "mmrl_logit_cos"),
    ("delta_sym_kl", "bayes_sym_kl", "mmrl_sym_kl"),
    ("delta_js_div", "bayes_js_div", "mmrl_js_div"),
    ("delta_logit_std_f", "bayes_logit_std_f", "mmrl_logit_std_f"),
    ("delta_mutual_info_f", "bayes_mutual_info_f", "mmrl_mutual_info_f"),
]:
    compare_df[name] = compare_df[a] - compare_df[b]

group_summary = pd.DataFrame({
    "bayes_better": compare_df[compare_df["bayes_better"]].mean(numeric_only=True),
    "bayes_worse": compare_df[compare_df["bayes_worse"]].mean(numeric_only=True),
    "same_correct": compare_df[compare_df["same_correct"]].mean(numeric_only=True),
    "same_wrong": compare_df[compare_df["same_wrong"]].mean(numeric_only=True),
}).T

group_summary[[
    "delta_conf_f",
    "delta_margin_f",
    "delta_feat_cos",
    "delta_logit_cos",
    "delta_sym_kl",
    "delta_js_div",
    "delta_logit_std_f",
    "delta_mutual_info_f",
]]


## 自动结论

In [ ]:

def sign_word(x, pos="上升", neg="下降", zero="基本不变", thr=1e-6):
    if x > thr:
        return pos
    if x < -thr:
        return neg
    return zero

delta_row = delta["Bayes - MMRL"]

report = f'''
BayesMMRL 相对 MMRL 的自动结论摘要：

1. acc_f: {delta_row["acc_f"]:+.4f}，说明 BayesMMRL 在最终判别上{sign_word(delta_row["acc_f"], "更好", "更差")}。
2. complementarity: {delta_row["complementarity"]:+.4f}，说明两个分支互补关系{sign_word(delta_row["complementarity"], "增强", "减弱")}。
3. feat_cos: {delta_row["feat_cos"]:+.4f}；logit_cos: {delta_row["logit_cos"]:+.4f}。
4. sym_kl: {delta_row["sym_kl"]:+.4f}；js_div: {delta_row["js_div"]:+.4f}。
5. hurt_best_branch: {delta_row["hurt_best_branch"]:+.4f}。
6. mutual_info_f: {delta_row["mutual_info_f"]:+.4f}。
'''
print(report)


## 导出结果

In [ ]:

OUTPUT_DIR = Path("./analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mmrl_df.to_csv(OUTPUT_DIR / "mmrl_feature_analysis.csv", index=False)
bayes_df.to_csv(OUTPUT_DIR / "bayes_feature_analysis.csv", index=False)
summary_table.to_csv(OUTPUT_DIR / "summary_table.csv", index=False)
delta.to_csv(OUTPUT_DIR / "delta_table.csv")

print("Saved to:", OUTPUT_DIR.resolve())
